# ML441 ASSIGNMENT 4: ENSEMBLE LEARNING FOR UNSW DATA


In [ ]:

#imports
from importlib.metadata import version
from itertools import combinations
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn import set_config
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (AdaBoostClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_recall_fscore_support)
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from threadpoolctl import threadpool_limits

#global variables and computational settings

SEED, THREADS, KNN_BATCH_SIZE = 0, 2, 1000
assert version('scikit-learn') == '1.7.1', 'Use scikit-learn 1.7.1 for the original group splits'
set_config(working_memory=256)
plt.rcParams.update({'font.family': 'serif', 'font.size': 10, 'figure.figsize': (7.1, 4.4),
                     'axes.spines.top': False, 'axes.spines.right': False})

In [ ]:
#SETP INPUT AND OUTPUT 
DATA = Path("data")  # Use Path(".") if the CSVs are in the current folder.
OUTPUT = Path("submission_outputs")
RESULTS, FIGURES = OUTPUT / "results", OUTPUT / "figures"

for folder in [RESULTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)

#LOADS DATA
df = pd.read_csv(DATA / "networkTraffic.csv", na_values="?")
df = df.dropna(subset=["attack_cat"])

attack_map = pd.read_csv(DATA / "attack_category_map.csv").sort_values("Mapping")
class_labels = attack_map["Mapping"].to_numpy()
class_names = attack_map["Attack Category Name"].tolist()
normal_code = int(
    attack_map.loc[attack_map["Attack Category Name"].eq("Normal"), "Mapping"].item()
)

#PREDICTORS, TARGET AND DUPLICATE GROUPS
categorical = ["proto", "state", "service"]
X = df.drop(columns=["id", "attack_cat"])
y = df["attack_cat"]
numeric = X.columns.difference(categorical).tolist()
duplicate_group = pd.util.hash_pandas_object(X, index=False)

In [ ]:
##equal predictor rows always stay together.
eligible_index = df.index
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
development_pos, test_pos = next(outer.split(X, y, groups=duplicate_group))
development_index, test_index = eligible_index[development_pos], eligible_index[test_pos]
tuning_splitter = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
_, tuning_pos = next(tuning_splitter.split(X.loc[development_index], y.loc[development_index],
                                         groups=duplicate_group.loc[development_index]))
tuning_index = development_index[tuning_pos]
remaining_index = development_index[~development_index.isin(tuning_index)]
comparison_splitter = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
_, comparison_pos = next(comparison_splitter.split(X.loc[remaining_index], y.loc[remaining_index],
                                                  groups=duplicate_group.loc[remaining_index]))
comparison_index = remaining_index[comparison_pos]
heldout_index, _ = train_test_split(test_index, train_size=10000, stratify=y.loc[test_index], random_state=SEED)

assert [len(df), len(development_index), len(test_index), len(tuning_index), len(comparison_index)] == [257673, 207001, 50672, 29365, 30315]
for a, b in combinations([tuning_index, comparison_index, test_index], 2):
    assert set(duplicate_group.loc[a]).isdisjoint(duplicate_group.loc[b])

In [ ]:
tuning_folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(
    X.loc[tuning_index], y.loc[tuning_index], groups=duplicate_group.loc[tuning_index]))
comparison_folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(
    X.loc[comparison_index], y.loc[comparison_index], groups=duplicate_group.loc[comparison_index]))
manifest = pd.DataFrame({'duplicate_group': duplicate_group.astype(str), 'role': 'development_unused',
                         'validation_fold': 0, 'heldout_order': -1})
manifest.loc[test_index, 'role'] = 'outer_test'
manifest.loc[heldout_index, 'heldout_order'] = np.arange(len(heldout_index))
#


for role, indices, folds in [('tuning', tuning_index, tuning_folds), ('comparison', comparison_index, comparison_folds)]:
    manifest.loc[indices, 'role'] = role
    for fold, (tr, va) in enumerate(folds, 1):
        assert set(duplicate_group.loc[indices[tr]]).isdisjoint(duplicate_group.loc[indices[va]])
        assert set(y.loc[indices[tr]]) == set(y.loc[indices[va]]) == set(class_labels)
        manifest.loc[indices[va], 'validation_fold'] = fold
manifest.rename_axis('original_row_index').to_csv(RESULTS / 'split_manifest.csv')

In [ ]:



knn_numeric = X.columns.difference(categorical + ['sloss', 'dloss', 'ct_ftp_cmd']).tolist()
knn_preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('scale', StandardScaler()),
        ('float32', FunctionTransformer(np.asarray, kw_args={'dtype': np.float32}, feature_names_out='one-to-one')),
    ]), knn_numeric),
    ('categorical', Pipeline([
        ('missing', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OneHotEncoder(handle_unknown='ignore', dtype=np.float32)),
    ]), categorical),
])
tree_preprocessor = ColumnTransformer([
    ('numeric', 'passthrough', numeric),
    ('categorical', Pipeline([
        ('missing', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), categorical),
])


ensemble_preprocessor = ColumnTransformer([
    ('numeric', FunctionTransformer(np.asarray, kw_args={'dtype': np.float32}, feature_names_out='one-to-one'), numeric),
    ('categorical', Pipeline([
        ('missing', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.float32)),
    ]), categorical),
], sparse_threshold=0)

In [ ]:
historical_knn = Pipeline([('prepare', knn_preprocessor), ('model', KNeighborsClassifier(
    n_neighbors=9, weights='distance', p=1, algorithm='brute', n_jobs=1))])
historical_tree = Pipeline([('prepare', tree_preprocessor), ('model', DecisionTreeClassifier(
    criterion='entropy', ccp_alpha=0.0005055995312775079, class_weight='balanced', random_state=SEED))])
candidates = {
    'Random forest': Pipeline([('prepare', ensemble_preprocessor), ('model', RandomForestClassifier(
        n_estimators=100, criterion='gini', max_depth=None, bootstrap=True, class_weight=None, n_jobs=2, random_state=SEED))]),
    'Extra trees': Pipeline([('prepare', ensemble_preprocessor), ('model', ExtraTreesClassifier(
        n_estimators=100, criterion='gini', max_depth=None, bootstrap=False, class_weight=None, n_jobs=2, random_state=SEED))]),
    'AdaBoost': Pipeline([('prepare', ensemble_preprocessor), ('model', AdaBoostClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED, class_weight=None), n_estimators=100, random_state=SEED))]),
    'Histogram gradient boosting': Pipeline([('prepare', ensemble_preprocessor), ('model', HistGradientBoostingClassifier(
        max_iter=100, min_samples_leaf=20, l2_regularization=0.0, early_stopping=False,
        categorical_features=None, class_weight=None, random_state=SEED))]),
}
screen_grids = {
    'Random forest': {'model__max_features': ['sqrt', 0.5], 'model__min_samples_leaf': [1, 5], 'model__n_estimators': [100]},
    'Extra trees': {'model__max_features': ['sqrt', 0.5], 'model__min_samples_leaf': [1, 5], 'model__n_estimators': [100]},
    'AdaBoost': {'model__learning_rate': [0.1, 1.0], 'model__estimator__max_depth': [1, 3], 'model__n_estimators': [100]},
    'Histogram gradient boosting': {'model__learning_rate': [0.05, 0.1], 'model__max_leaf_nodes': [15, 31], 'model__max_iter': [100]},
}

In [ ]:
#DEFINE FUNCTUIONS FOR CODE REPRODUCIBILITY
#GRID SEARCH AND EVALUATION FUNCTIONS NEEDED HERE

def metric_summary(truth, prediction):
    return {'macro_f1': f1_score(truth, prediction, labels=class_labels, average='macro', zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(truth, prediction),
            'accuracy': accuracy_score(truth, prediction)}

def predict_in_batches(pipeline, inputs):
    if isinstance(pipeline.named_steps['model'], KNeighborsClassifier):
        return np.concatenate([pipeline.predict(inputs.iloc[start:start + KNN_BATCH_SIZE])
                               for start in range(0, len(inputs), KNN_BATCH_SIZE)])
    return pipeline.predict(inputs)

def summarise_folds(rows):
    keys = ['stage', 'family', 'configuration_id', 'parameter_json', 'weighting']
    fields = ['macro_f1', 'balanced_accuracy', 'accuracy', 'train_macro_f1',
              'fit_seconds', 'prediction_seconds']
    summary = pd.DataFrame(rows).groupby(keys, sort=False)[fields].agg(['mean', 'std']).reset_index()
    summary.columns = ['_'.join(filter(None, col)) for col in summary.columns]
    return summary

def select_best(summary):
    tied = summary.loc[summary.macro_f1_mean.ge(summary.macro_f1_mean.max() - 1e-12)]
    return tied.sort_values(['prediction_seconds_mean', 'fit_seconds_mean', 'family', 'configuration_id']).iloc[0]

def evaluate_folds(pipeline, indices, folds, *, stage, family, params, weighting='unweighted', training_score=True):
    rows = []
    parameter_json = json.dumps(params, sort_keys=True)
    for fold, (tr, va) in enumerate(folds, 1):
        fitted = clone(pipeline).set_params(**params)
        X_train, y_train = X.loc[indices[tr]], y.loc[indices[tr]]
        X_val, y_val = X.loc[indices[va]], y.loc[indices[va]]
        fit_kwargs = {'model__sample_weight': compute_sample_weight('balanced', y_train)} if weighting == 'balanced' else {}
        with threadpool_limits(limits=THREADS):
            start = perf_counter()
            fitted.fit(X_train, y_train, **fit_kwargs)
            fit_seconds = perf_counter() - start
            start = perf_counter()
            prediction = predict_in_batches(fitted, X_val)
            prediction_seconds = perf_counter() - start
            train_f1 = metric_summary(y_train, predict_in_batches(fitted, X_train))['macro_f1'] if training_score else np.nan
        precision, recall, _, support = precision_recall_fscore_support(y_val, prediction, labels=class_labels, zero_division=0)
        normal = y_val.to_numpy() == normal_code
        rows.append({'stage': stage, 'family': family, 'configuration_id': family + ':' + parameter_json,
                     'parameter_json': parameter_json, 'weighting': weighting, 'fold': fold, 'reused_from_stage': None,
                     **metric_summary(y_val, prediction), 'train_macro_f1': train_f1,
                     'fit_seconds': fit_seconds, 'prediction_seconds': prediction_seconds,
                     'class_precision_json': json.dumps(precision.tolist()), 'class_recall_json': json.dumps(recall.tolist()),
                     'class_support_json': json.dumps(support.tolist()),
                     'normal_as_attack_rate': np.mean(prediction[normal] != normal_code),
                     'attack_as_normal_rate': np.mean(prediction[~normal] == normal_code)})
        print(f'{stage} | {family} | fold {fold}: macro-F1={rows[-1]["macro_f1"]:.4f}', flush=True)
    return rows

def run_grid(family, grid, stage, existing_rows=()):
    rows = []
    for params in ParameterGrid(grid):
        parameter_json = json.dumps(params, sort_keys=True)
        previous = [r for r in existing_rows if r['family'] == family and r['parameter_json'] == parameter_json
                    and r['weighting'] == 'unweighted']
        if previous:
            assert len(previous) == 5
            rows.extend([{**r, 'stage': stage, 'reused_from_stage': r['stage']} for r in previous])
        else:
            rows.extend(evaluate_folds(candidates[family], tuning_index, tuning_folds,
                                       stage=stage, family=family, params=params))
    return rows

In [ ]:
tuning_rows = []
for family, grid in screen_grids.items():
    tuning_rows.extend(run_grid(family, grid, 'screening'))
screen_summary = summarise_folds(tuning_rows)
family_summary = pd.DataFrame([select_best(g) for _, g in screen_summary.groupby('family')])
selected_family = select_best(family_summary).family
display(family_summary)

In [ ]:
def save_figure(name):
    plt.gcf().tight_layout()
    plt.savefig(FIGURES / name, dpi=300, bbox_inches='tight')
    plt.show()
fig, ax = plt.subplots()
positions = np.arange(len(family_summary))
ax.errorbar(positions, family_summary.macro_f1_mean, yerr=family_summary.macro_f1_std, fmt='o', capsize=4)
ax.set(xticks=positions, xticklabels=family_summary.family.str.replace('Histogram gradient boosting', 'Histogram gradient\nboosting', regex=False), ylabel='Mean tuning macro-F1')
ax.tick_params(axis='x', labelrotation=18)
save_figure('ensemble_screening.png')

In [ ]:
#rrefine only the tuning-selected family, reusing its screening fits.
refinement_grids = {
    'Random forest': {**screen_grids['Random forest'], 'model__n_estimators': [100, 300, 500]},
    'Extra trees': {**screen_grids['Extra trees'], 'model__n_estimators': [100, 300, 500]},
    'AdaBoost': {**screen_grids['AdaBoost'], 'model__n_estimators': [100, 200, 300]},
    'Histogram gradient boosting': {**screen_grids['Histogram gradient boosting'], 'model__max_iter': [100, 200, 300]},
}
refinement_rows = run_grid(selected_family, refinement_grids[selected_family], 'refinement',
                           existing_rows=[r for r in tuning_rows if r['stage'] == 'screening'])
tuning_rows.extend(refinement_rows)
refinement_summary = summarise_folds(refinement_rows)
structural_winner = select_best(refinement_summary)
selected_params = json.loads(structural_winner.parameter_json)
display(refinement_summary)

"""
refinement_grids = {
    'Random forest': {**screen_grids['Random forest'], 'model__n_estimators': [100, 300, 500]},
    'Extra trees': {**screen_grids['Extra trees'], 'model__n_estimators': [100, 300, 500]},
    'AdaBoost': {**screen_grids['AdaBoost'], 'model__n_estimators': [100, 200, 300]},
    'Histogram gradient boosting': {**screen_grids['Histogram gradient boosting'], 'model__max_iter': [100, 200, 300]},
}
refinement_rows = run_grid(selected_family, refinement_grids[selected_family], 'refinement',
                           existing_rows=[r for r in tuning_rows if r['stage'] == 'screening'])
tuning_rows.extend(refinement_rows)
refinement_summary = summarise_folds(refinement_rows)
structural_winner = select_best(refinement_summary)
selected_params = json.loads(structural_winner.parameter_json)
display(refinement_summary"""

In [ ]:
#plot ensemble size and training-validation gaps.
size_key = 'model__max_iter' if selected_family == 'Histogram gradient boosting' else 'model__n_estimators'
curve = refinement_summary.copy()
curve['size'] = curve.parameter_json.map(lambda s: json.loads(s)[size_key])
parameter_labels = {'model__max_features': 'Features', 'model__min_samples_leaf': 'Min. leaf',
                    'model__learning_rate': 'Rate', 'model__max_leaf_nodes': 'Max. leaves',
                    'model__estimator__max_depth': 'Depth'}
curve['other_settings'] = curve.parameter_json.map(lambda s: ', '.join(
    f'{parameter_labels[k]}={v}' for k, v in json.loads(s).items() if k != size_key))
fig, ax = plt.subplots(figsize=(7.1, 4.6))
for setting, rows in curve.groupby('other_settings'):
    rows = rows.sort_values('size')
    ax.errorbar(rows['size'], rows.macro_f1_mean, yerr=rows.macro_f1_std, marker='o', capsize=3, label=setting)
ax.set(xlabel='Boosting iterations' if 'max_iter' in size_key else 'Number of estimators',
       ylabel='Mean tuning macro-F1', xticks=sorted(curve['size'].unique()))
ax.legend(fontsize=10, loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2)
save_figure('finalist_tuning.png')
fig, axes = plt.subplots(2, 2, figsize=(7.1, 5.7), sharex=True, sharey=True)
for ax, (setting, rows) in zip(axes.flat, curve.groupby('other_settings')):
    rows = rows.sort_values('size')
    ax.plot(rows['size'], rows.train_macro_f1_mean, 'o-', label='Training')
    ax.plot(rows['size'], rows.macro_f1_mean, 's--', label='Validation')
    ax.set_title(setting, fontsize=10)
    ax.set_xticks(sorted(curve['size'].unique()))
    ax.set_xlabel('Ensemble size')
    ax.set_ylabel('Macro-averaged F1')
axes.flat[0].legend(fontsize=10)
save_figure('training_validation_gap.png')

_______________________

In [ ]:


unweighted_rows = [{**r, 'stage': 'weighting', 'reused_from_stage': 'refinement'}
                   for r in refinement_rows if r['parameter_json'] == structural_winner.parameter_json]
weighting_rows = unweighted_rows + evaluate_folds(candidates[selected_family], tuning_index, tuning_folds,
    stage='weighting', family=selected_family, params=selected_params, weighting='balanced')
tuning_rows.extend(weighting_rows)
weighting_summary = summarise_folds(weighting_rows)
scores = weighting_summary.set_index('weighting').macro_f1_mean
selected_weighting = 'balanced' if scores.balanced > scores.unweighted else 'unweighted'
frozen_ensemble = clone(candidates[selected_family]).set_params(**selected_params)
selection = {'family': selected_family, 'pipeline_parameters': selected_params, 'weighting': selected_weighting, 'seed': SEED}
(RESULTS / 'selected_configuration.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
pd.DataFrame(tuning_rows).to_csv(RESULTS / 'tuning_folds.csv', index=False)
summarise_folds(tuning_rows).to_csv(RESULTS / 'tuning_summary.csv', index=False)
display(weighting_summary)
print('Selected:', selection)

In [ ]:
# Independent comparison: all models use the same five folds.
matched_tree = Pipeline([('prepare', ensemble_preprocessor), ('model', DecisionTreeClassifier(
    criterion='entropy', ccp_alpha=0.0005055995312775079, class_weight=None, random_state=SEED))])
comparison_models = {'Historical kNN': (historical_knn, 'unweighted'),
                     'Historical tree': (historical_tree, 'class_weight_balanced'),
                     'Matched tree': (matched_tree, selected_weighting),
                     'Selected ensemble': (frozen_ensemble, selected_weighting)}
comparison_rows = []
for family, (pipeline, weighting) in comparison_models.items():
    comparison_rows.extend(evaluate_folds(pipeline, comparison_index, comparison_folds,
                           stage='comparison', family=family, params={}, weighting=weighting, training_score=False))
pd.DataFrame(comparison_rows).to_csv(RESULTS / 'comparison_folds.csv', index=False)
comparison_summary = summarise_folds(comparison_rows)
comparison_summary.to_csv(RESULTS / 'comparison_summary.csv', index=False)
paired = pd.DataFrame(comparison_rows).pivot(index='fold', columns='family', values='macro_f1')
display(comparison_summary.drop(columns=['train_macro_f1_mean', 'train_macro_f1_std']))
paired_differences = pd.DataFrame({name: paired['Selected ensemble'] - paired[name]
                                  for name in comparison_models if name != 'Selected ensemble'})
display(paired_differences.agg(['mean', 'std', 'min', 'max']))

In [ ]:
fig, ax = plt.subplots()
for family in comparison_models:
    ax.plot(paired.index, paired[family], 'o-', label=family)
ax.set(xlabel='Shared comparison fold', ylabel='Macro-averaged F1', xticks=paired.index)
ax.legend(fontsize=10)
save_figure('paired_comparison.png')

In [ ]:
# This 10000-row holdout was already evaluated in Assignment 2.
predictions = pd.DataFrame({'original_row_index': heldout_index, 'true_target': y.loc[heldout_index].to_numpy()})
final_timings = {}


for family, (pipeline, weighting) in comparison_models.items():
    fitted = clone(pipeline)
    fit_kwargs = {'model__sample_weight': compute_sample_weight('balanced', y.loc[comparison_index])} if weighting == 'balanced' else {}
    with threadpool_limits(limits=THREADS):
        start = perf_counter()
        fitted.fit(X.loc[comparison_index], y.loc[comparison_index], **fit_kwargs)
        fit_seconds = perf_counter() - start
        start = perf_counter()
        predictions[family] = predict_in_batches(fitted, X.loc[heldout_index])
        final_timings[family] = {'fit_seconds': fit_seconds, 'prediction_seconds': perf_counter() - start}
        
        
predictions.to_csv(RESULTS / 'heldout_predictions.csv', index=False)

In [ ]:
#calculate final metrics, class supports and Normal/attack error rates.



heldout_rows, per_class_rows = [], []
truth = predictions.true_target.to_numpy()
normal_mask = truth == normal_code
for family in comparison_models:
    prediction = predictions[family].to_numpy()
    precision, recall, f1, support = precision_recall_fscore_support(truth, prediction, labels=class_labels, zero_division=0)
    for label, name, p, r, f, n in zip(class_labels, class_names, precision, recall, f1, support):
        per_class_rows.append({'model': family, 'class_code': label, 'class_name': name,
                               'precision': p, 'recall': r, 'f1': f, 'support': n})
    normal_as_attack = int(np.sum(normal_mask & (prediction != normal_code)))
    attack_as_normal = int(np.sum(~normal_mask & (prediction == normal_code)))
    heldout_rows.append({'model': family, **metric_summary(truth, prediction), **final_timings[family],
                         'normal_as_attack_count': normal_as_attack, 'normal_support': int(normal_mask.sum()),
                         'normal_as_attack_rate': normal_as_attack / normal_mask.sum(),
                         'attack_as_normal_count': attack_as_normal, 'attack_support': int((~normal_mask).sum()),
                         'attack_as_normal_rate': attack_as_normal / (~normal_mask).sum()})
heldout_summary = pd.DataFrame(heldout_rows)
per_class_results = pd.DataFrame(per_class_rows)
heldout_summary.to_csv(RESULTS / 'heldout_summary.csv', index=False)
per_class_results.to_csv(RESULTS / 'per_class_results.csv', index=False)
display(heldout_summary)
display(per_class_results)

In [ ]:
recall_table = per_class_results.pivot(index='class_code', columns='model', values='recall').reindex(class_labels)
recall_table.index = class_names
ax = recall_table[list(comparison_models)].plot.bar(figsize=(7.1, 5.0), width=0.8)
ax.set(xlabel='True traffic class', ylabel='Held-out recall', ylim=(0, 1.05))
ax.set_xticklabels(class_names, rotation=40, ha='right')
ax.legend(fontsize=10, ncol=2)
save_figure('per_class_recall.png')
for family, filename in [('Selected ensemble', 'confusion_matrix_ensemble.png'),
                          ('Historical tree', 'confusion_matrix_tree.png')]:
    matrix = confusion_matrix(truth, predictions[family], labels=class_labels, normalize='true')
    fig, ax = plt.subplots(figsize=(7.1, 6.2))
    image = ax.imshow(matrix, cmap='Blues', vmin=0, vmax=1)
    ax.set(xticks=np.arange(10), yticks=np.arange(10), xticklabels=class_names, yticklabels=class_names,
           xlabel='Predicted traffic class', ylabel='True traffic class')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
    ax.tick_params(axis='y', labelsize=10)
    for row in range(10):
        for col in range(10):
            ax.text(col, row, f'{matrix[row, col]:.2f}', ha='center', va='center', fontsize=10,
                    color='white' if matrix[row, col] > 0.5 else 'black')
    fig.colorbar(image, ax=ax, shrink=0.8, label='Proportion of true class')
    save_figure(filename)

In [ ]:
assert len(tuning_rows) == 150 and len(comparison_rows) == 20
assert sum(r['reused_from_stage'] is not None for r in tuning_rows) == 25
assert per_class_results.groupby('model').support.sum().eq(len(heldout_index)).all()
print('Complete. Results and figures:', RESULTS.parent)